In [ ]:
import glob, os
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def find(patterns):
    for root in ["/kaggle/input", "."]:
        for p in patterns:
            hits = sorted(glob.glob(os.path.join(root, "**", p), recursive=True))
            if hits:
                return hits[0]
    raise FileNotFoundError(patterns)

train = pd.read_csv(find(["train.csv"]))
test = pd.read_csv(find(["test.csv", "test*.csv"]))
sample = pd.read_csv(find(["sample_submission.csv", "submission.csv"]))

def make_features(d):
    d = d.copy()
    d["route"] = d.source_city + ">" + d.destination_city
    d["log_dur"] = np.log(d.duration)
    d["dur2"] = d.duration ** 2
    d["log_days"] = np.log(d.days_left)
    days_bin = pd.cut(d.days_left, [0, 2, 4, 7, 14, 21, 30, 45], labels=False).astype(str)
    days_bin2 = pd.cut(d.days_left, [0, 3, 7, 14, 30, 45], labels=False).astype(str)
    dur_bin = pd.cut(d.duration, [0, 2, 3, 5, 7, 10, 15, 40], labels=False).astype(str)
    c = d.flight_class
    d["cls_air"] = c + "_" + d.airline
    d["cls_route"] = c + "_" + d.route
    d["cls_stops"] = c + "_" + d.stops
    d["cls_dep"] = c + "_" + d.departure_time
    d["cls_arr"] = c + "_" + d.arrival_time
    d["cls_days_bin"] = c + "_" + days_bin
    d["cls_flight"] = c + "_" + d.flight
    d["air_route"] = d.airline + "_" + d.route
    d["cls_air_stops"] = d.cls_air + "_" + d.stops
    d["cls_air_route"] = d.cls_air + "_" + d.route
    d["cls_flight_stops"] = d.cls_flight + "_" + d.stops
    d["cls_flight_dur"] = d.cls_flight + "_" + d.duration.round(1).astype(str)
    d["cls_route_dep"] = d.cls_route + "_" + d.departure_time
    d["cls_air_dep"] = d.cls_air + "_" + d.departure_time
    d["cls_air_days"] = d.cls_air + "_" + days_bin2
    d["cls_route_days"] = d.cls_route + "_" + days_bin2
    d["cls_flight_days"] = d.cls_flight + "_" + days_bin2
    d["dur_bin"] = c + "_" + d.stops + "_" + dur_bin
    return d

X_train, X_test = make_features(train), make_features(test)
y = train["price"].values.astype(float)

cat_cols = [
    "airline", "source_city", "destination_city", "departure_time", "arrival_time",
    "stops", "flight_class", "flight",
    "cls_air", "cls_route", "cls_stops", "cls_dep", "cls_arr", "cls_days_bin", "cls_flight",
    "air_route", "cls_air_stops", "cls_air_route", "cls_flight_stops", "cls_flight_dur",
    "cls_route_dep", "cls_air_dep", "cls_air_days", "cls_route_days", "cls_flight_days", "dur_bin",
]
num_cols = ["duration", "days_left", "log_dur", "log_days", "dur2"]

ALPHA, CLIP_Q, W_LOG = 5.0, 0.999, 0.3

def build():
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ])
    return Pipeline([("pre", pre), ("ridge", Ridge(alpha=ALPHA))])

def fit_predict(Xa, ya, Xb):
    y_clip = np.clip(ya, 0, np.quantile(ya, CLIP_Q))
    raw = build().fit(Xa, y_clip).predict(Xb)
    log = np.expm1(build().fit(Xa, np.log1p(y_clip)).predict(Xb))
    return np.clip((1 - W_LOG) * raw + W_LOG * log, 0, None)

oof = np.zeros(len(y))
for tr_idx, va_idx in KFold(5, shuffle=True, random_state=42).split(X_train):
    oof[va_idx] = fit_predict(X_train.iloc[tr_idx], y[tr_idx], X_train.iloc[va_idx])

print(f"5-fold CV R2   : {r2_score(y, oof):.4f}")
print(f"5-fold CV RMSE : {np.sqrt(np.mean((y - oof) ** 2)):.2f}")

pred = fit_predict(X_train, y, X_test)
sub = pd.DataFrame({
    sample.columns[0]: test["id"].values,
    sample.columns[1]: pred
})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("submission shape:", sub.shape, "| columns:", list(sub.columns))
print(sub.head())